In [ ]:
import cv2
from djitellopy import tello
import pygame
import time
import os

pygame.init()
win = pygame.display.set_mode((400, 400)) 

if not os.path.exists("Surveillance Photos"):
    os.makedirs("Surveillance Photos")

drone = tello.Tello()
drone.connect()

try:
    print(f"Battery = {drone.get_battery()}")
except BaseException:
    print(f"Battery = 100 %")  

drone.streamon()
time.sleep(2) 
drone.takeoff()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            drone.land()
            pygame.quit()
            exit()

    frame = drone.get_frame_read().frame
    
    img = cv2.resize(frame, (360, 240))
    cv2.imshow("Tello Live Surveillance Stream", img)
    cv2.waitKey(1) 

    keys = pygame.key.get_pressed()
    lr,fb,ud,yaw = 0,0,0,0 
    speed = 50
    if keys[pygame.K_w]: fb = speed
    elif keys[pygame.K_s]: fb = -speed

    if keys[pygame.K_a]: lr = -speed
    elif keys[pygame.K_d]: lr = speed

    if keys[pygame.K_UP]: ud = speed
    elif keys[pygame.K_DOWN]: ud = -speed

    if keys[pygame.K_LEFT]: yaw = -speed
    elif keys[pygame.K_RIGHT]: yaw = speed

    drone.send_rc_control(lr, ud, fb, yaw)

    if keys[pygame.K_SPACE]:
        filename = f"Surveillance Photos/photo_{int(time.time())}.jpg"
        cv2.imwrite(filename, frame)
        print(f"Successfully Saved the Frame!")
        time.sleep(0.3)

    if keys[pygame.K_ESCAPE]:
        drone.send_rc_control(0, 0, 0, 0)
        drone.land()
        break

drone.streamoff()
cv2.destroyAllWindows()
pygame.quit()
